# 10. 파이썬 기초 - 날짜와 시간 다루기

스크래핑한 날짜는 대부분 **문자열** 입니다.
`"26/08/2014"` 는 문자열일 뿐이라 뺄셈도, 연도 추출도, 정렬도 되지 않습니다.

**날짜형으로 바꾸는 순간** 이 모든 것이 가능해집니다.

**다루는 내용**
1. datetime / timedelta 기본
2. 문자열 ↔ 날짜 (strptime / strftime)
3. pandas `to_datetime` — 열 전체를 한 번에
4. `.dt` 접근자 — 연·월·요일 추출
5. 나이 계산 — K-Pop 분석의 실제 문제
6. 기간별 집계 — 연도별 추이
7. 종합 실습 — K-Pop 아이돌 데이터

In [1]:
from datetime import datetime, date, timedelta
import pandas as pd

print('오늘:', date.today())
print('지금:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

오늘: 2026-08-14
지금: 2026-08-14 00:09:51


## 1. datetime / timedelta 기본

- `date` : 날짜만 (연월일)
- `datetime` : 날짜 + 시각
- `timedelta` : **기간** (두 날짜의 차이, 또는 더하고 뺄 기간)

In [2]:
d1 = date(2014, 8, 26)
d2 = date(2020, 1, 1)

# 날짜끼리 빼면 timedelta(기간) 이 나온다
gap = d2 - d1
print('두 날짜의 차이:', gap)
print('일수만:', gap.days)

print()
# 날짜 + 기간 = 새로운 날짜
print('100일 후:', d1 + timedelta(days=100))
print('1주 전  :', d1 - timedelta(weeks=1))

print()
# 개별 요소 꺼내기
print('연:', d1.year, '/ 월:', d1.month, '/ 일:', d1.day)
print('요일 번호:', d1.weekday(), '(0=월요일, 6=일요일)')

두 날짜의 차이: 1954 days, 0:00:00
일수만: 1954

100일 후: 2014-12-04
1주 전  : 2014-08-19

연: 2014 / 월: 8 / 일: 26
요일 번호: 1 (0=월요일, 6=일요일)


## 2. 문자열 ↔ 날짜 (strptime / strftime)

이름이 헷갈리기 쉬운데, 마지막 글자로 기억하면 됩니다.

- `strptime` : **p = parse** → 문자열을 **읽어서** 날짜로
- `strftime` : **f = format** → 날짜를 **꾸며서** 문자열로

| 기호 | 의미 | 예 |
|---|---|---|
| `%Y` | 연도 4자리 | 2014 |
| `%m` | 월 2자리 | 08 |
| `%d` | 일 2자리 | 26 |
| `%H:%M:%S` | 시:분:초 | 14:30:00 |

In [3]:
# 문자열 → 날짜 : 문자열의 '생김새' 를 형식으로 알려줘야 한다
s = '26/08/2014'
parsed = datetime.strptime(s, '%d/%m/%Y')    # 일/월/연 순서
print('문자열:', s, type(s))
print('날짜형:', parsed, type(parsed))

print()
# 날짜 → 문자열 : 원하는 모양으로 출력
print('%Y-%m-%d      :', parsed.strftime('%Y-%m-%d'))
print('%Y년 %m월 %d일 :', parsed.strftime('%Y년 %m월 %d일'))
print('%y/%m         :', parsed.strftime('%y/%m'))

문자열: 26/08/2014 <class 'str'>
날짜형: 2014-08-26 00:00:00 <class 'datetime.datetime'>

%Y-%m-%d      : 2014-08-26
%Y년 %m월 %d일 : 2014년 08월 26일
%y/%m         : 14/08


In [4]:
# 형식이 맞지 않으면 에러가 난다 → 스크래핑 데이터에서 매우 흔한 상황
bad = '2014-08-26'
try:
    datetime.strptime(bad, '%d/%m/%Y')
except ValueError as exp:
    print('에러 발생:', exp)
    print('→ 문자열 생김새와 형식 문자열이 반드시 일치해야 한다')

에러 발생: time data '2014-08-26' does not match format '%d/%m/%Y'
→ 문자열 생김새와 형식 문자열이 반드시 일치해야 한다


## 3. pandas `to_datetime` — 열 전체를 한 번에 ⭐

`strptime` 은 값 하나씩 처리합니다. 표의 열 전체는 `pd.to_datetime()` 으로 한 번에 바꿉니다.

**중요 옵션**

| 옵션 | 하는 일 |
|---|---|
| `format='%d/%m/%Y'` | 형식을 명시 (가장 빠르고 안전) |
| `dayfirst=True` | 일/월/연 순서임을 알림 |
| `errors='coerce'` | 변환 실패한 값을 에러 대신 **NaT**(빈 날짜)로 |

In [5]:
df = pd.DataFrame({
    'name':  ['A', 'B', 'C', 'D'],
    'debut': ['26/08/2014', '31/10/2015', '11/10/2017', '이상한값'],
})

print('변환 전 dtype:', df['debut'].dtype)   # object = 문자열

# errors='coerce' : 변환 못 하는 값은 NaT 로 만들고 계속 진행
df['debut'] = pd.to_datetime(df['debut'], dayfirst=True, errors='coerce')

print('변환 후 dtype:', df['debut'].dtype)   # datetime64[ns]
print()
print(df)
print('\n변환 실패(NaT) 개수:', df['debut'].isna().sum())

변환 전 dtype: object
변환 후 dtype: datetime64[ns]

  name      debut
0    A 2014-08-26
1    B 2015-10-31
2    C 2017-10-11
3    D        NaT

변환 실패(NaT) 개수: 1


In [6]:
# 날짜형이 되면 '비교' 와 '정렬' 이 제대로 동작한다
print('2015년 이후 데뷔:')
print(df[df['debut'] >= '2015-01-01'])

print()
print('데뷔일 순 정렬:')
print(df.sort_values('debut'))

# 문자열이었다면 '31/10/2015' < '26/08/2014' 처럼
# 글자 순서로 비교되어 엉뚱한 결과가 나온다

2015년 이후 데뷔:
  name      debut
1    B 2015-10-31
2    C 2017-10-11

데뷔일 순 정렬:
  name      debut
0    A 2014-08-26
1    B 2015-10-31
2    C 2017-10-11
3    D        NaT


## 4. `.dt` 접근자 — 연·월·요일 추출

문자열 열에 `.str` 을 붙였듯이, 날짜 열에는 `.dt` 를 붙입니다.

In [7]:
df2 = df.dropna(subset=['debut']).copy()    # NaT 행 제외

df2['year']    = df2['debut'].dt.year
df2['month']   = df2['debut'].dt.month
df2['weekday'] = df2['debut'].dt.day_name()      # 요일 이름
df2['ym']      = df2['debut'].dt.strftime('%Y-%m')   # 원하는 문자열로

print(df2)

print()
print('사용 가능한 것들: .dt.year .dt.month .dt.day .dt.hour')
print('                 .dt.dayofweek .dt.day_name() .dt.quarter .dt.strftime()')

  name      debut  year  month    weekday       ym
0    A 2014-08-26  2014      8    Tuesday  2014-08
1    B 2015-10-31  2015     10   Saturday  2015-10
2    C 2017-10-11  2017     10  Wednesday  2017-10

사용 가능한 것들: .dt.year .dt.month .dt.day .dt.hour
                 .dt.dayofweek .dt.day_name() .dt.quarter .dt.strftime()


## 5. 나이 계산 — K-Pop 분석의 실제 문제 

"생일이 지났는지" 를 따져야 해서 단순히 연도만 빼면 틀립니다.

```python
나이 = 올해 - 태어난해 - (생일이 아직 안 지났으면 1)
```

`00python_basic.ipynb` 에 나왔던 **튜플 비교** 가 여기서 쓰입니다.
`(month, day)` 를 튜플로 묶어 비교하면 월·일을 한 번에 비교할 수 있습니다.

In [8]:
# 튜플 비교 복습: 앞 요소부터 차례로 비교한다
print('(12, 15) < (1, 1)  :', (12, 15) < (1, 1))    # 12 > 1 이므로 False
print('(12, 15) < (12, 17):', (12, 15) < (12, 17))  # 12==12 → 15 < 17 이므로 True

print()
print('→ True(1) / False(0) 는 숫자처럼 뺄셈에 쓸 수 있다')
print('   True 는', int(True), ', False 는', int(False))

(12, 15) < (1, 1)  : False
(12, 15) < (12, 17): True

→ True(1) / False(0) 는 숫자처럼 뺄셈에 쓸 수 있다
   True 는 1 , False 는 0


In [8]:
def calc_age(birth, today=None):
    """만 나이를 계산한다.

    Args:
        birth (date): 생년월일
        today (date): 기준일. 생략하면 오늘

    Returns:
        int: 만 나이
    """
    today = today or date.today()
    # (월, 일) 튜플 비교로 '생일이 아직 안 지났는지' 판정
    # 아직 안 지났으면 True(1) 가 되어 1살이 빠진다
    not_yet = (today.month, today.day) < (birth.month, birth.day)
    return today.year - birth.year - not_yet


기준일 = date(2024, 6, 15)
print('1997-09-10 생 →', calc_age(date(1997, 9, 10), 기준일), '세  (생일 전)')
print('1997-03-10 생 →', calc_age(date(1997, 3, 10), 기준일), '세  (생일 지남)')
print('1997-06-15 생 →', calc_age(date(1997, 6, 15), 기준일), '세  (생일 당일)')

1997-09-10 생 → 26 세  (생일 전)
1997-03-10 생 → 27 세  (생일 지남)
1997-06-15 생 → 27 세  (생일 당일)


## 6. 기간별 집계 — 연도별 추이

날짜형으로 바꿔두면 **연도별·월별 집계** 가 한 줄로 끝납니다.

In [ ]:
logs = pd.DataFrame({
    'date':  pd.to_datetime([
        '2023-01-15', '2023-01-20', '2023-02-03',
        '2023-02-11', '2024-01-05', '2024-03-22',
    ]),
    'views': [100, 150, 200, 120, 300, 250],
})
logs

,date,views
0,2023-01-15,100
1,2023-01-20,150
2,2023-02-03,200
3,2023-02-11,120
4,2024-01-05,300
5,2024-03-22,250


In [10]:

# 연도별 집계 — .dt.year 로 그룹
print('연도별 합계:')
logs.groupby(logs['date'].dt.year)['views'].sum()


연도별 합계:


date
2023    570
2024    550
Name: views, dtype: int64

In [11]:

print()
# 월별 집계 — strftime 으로 'YYYY-MM' 문자열 그룹
print('월별 합계:')
logs.groupby(logs['date'].dt.strftime('%Y-%m'))['views'].sum()


월별 합계:


date
2023-01    250
2023-02    320
2024-01    300
2024-03    250
Name: views, dtype: int64

In [ ]:
# resample: 날짜를 인덱스로 두면 'M'(월), 'Y'(연) 단위 재집계가 가능
# set_index() 함수는 date 컬럼을 인덱스로 설정
ts = logs.set_index('date')
ts

,views
date,
2023-01-15,100
2023-01-20,150
2023-02-03,200
2023-02-11,120
2024-01-05,300
2024-03-22,250


In [13]:

print('월 단위 resample:')
ts['views'].resample('ME').sum()     # ME = Month End


월 단위 resample:


date
2023-01-31    250
2023-02-28    320
2023-03-31      0
2023-04-30      0
2023-05-31      0
2023-06-30      0
2023-07-31      0
2023-08-31      0
2023-09-30      0
2023-10-31      0
2023-11-30      0
2023-12-31      0
2024-01-31    300
2024-02-29      0
2024-03-31    250
Freq: ME, Name: views, dtype: int64

In [ ]:

print()
print('→ 데이터가 없는 달도 0 으로 채워져 나온다는 점이 groupby 와 다르다')

## 7. 종합 실습 — K-Pop 아이돌 데이터

실제 데이터로 **문자열 → 날짜형 → 나이 계산 → 집계** 전 과정을 수행합니다.

In [17]:
idol = pd.read_csv('../data/kpopidolsv3.csv')
print('행 x 열:', idol.shape)
print()
print(idol[['Stage Name', 'Date of Birth', 'Debut', 'Group', 'Company']].head(3).to_string(index=False))
print()
print('Date of Birth 원본 dtype:', idol['Date of Birth'].dtype, '← 문자열')

행 x 열: (1778, 16)

Stage Name Date of Birth      Debut     Group Company
     2Soul    10/09/1997 26/08/2014 7 O'clock  Jungle
       A.M    31/12/1996  9/07/2019 Limitless     ONO
       Ace    28/08/1992 31/10/2015       VAV  A team

Date of Birth 원본 dtype: object ← 문자열


In [18]:
# 날짜 두 열을 한 번에 변환 (dd/mm/yyyy 형식이므로 dayfirst=True)
for col in ['Date of Birth', 'Debut']:
    idol[col] = pd.to_datetime(idol[col], dayfirst=True, errors='coerce')

print('변환 후 dtype:')
print(idol[['Date of Birth', 'Debut']].dtypes)

print()
print('변환 실패(NaT) 개수:')
print(idol[['Date of Birth', 'Debut']].isna().sum())

변환 후 dtype:
Date of Birth    datetime64[ns]
Debut            datetime64[ns]
dtype: object

변환 실패(NaT) 개수:
Date of Birth      2
Debut            153
dtype: int64


In [19]:
# 데뷔 나이 = 데뷔일 - 생년월일 (둘 다 날짜형이므로 뺄셈 가능)
valid = idol.dropna(subset=['Date of Birth', 'Debut']).copy()

# timedelta 를 연 단위로 환산 (.dt.days 로 일수를 꺼내 365.25 로 나눔)
valid['debut_age'] = ((valid['Debut'] - valid['Date of Birth']).dt.days / 365.25).round(1)

print('데뷔 나이 통계:')
print(valid['debut_age'].describe().round(1))

print()
print('가장 어린 나이에 데뷔한 5명:')
print(valid.nsmallest(5, 'debut_age')[['Stage Name', 'Group', 'debut_age']].to_string(index=False))

데뷔 나이 통계:
count    1623.0
mean       19.4
std         3.4
min        -2.8
25%        17.6
50%        19.4
75%        21.1
max        33.2
Name: debut_age, dtype: float64

가장 어린 나이에 데뷔한 5명:
Stage Name        Group  debut_age
      Hari Girls' World       -2.8
      A-ra Girls' World       -1.1
    Vivian        Gate9        0.1
      Yoon        Gate9        1.6
     Kyrin Girls' World        2.3


In [20]:
# 연도별 데뷔 인원 추이
by_year = valid['Debut'].dt.year.value_counts().sort_index()

print('최근 10년 데뷔 인원:')
by_year.tail(10)


최근 10년 데뷔 인원:


Debut
2014     91
2015    100
2016    114
2017    156
2018    116
2019    156
2020    200
2021    120
2022    178
2023     39
Name: count, dtype: int64

In [21]:

print()
# 소속사별 평균 데뷔 나이 (인원 20명 이상만)
company = valid.groupby('Company')['debut_age'].agg(['count', 'mean'])
company = company[company['count'] >= 20].sort_values('mean')
print('소속사별 평균 데뷔 나이 (20명 이상):')
print(company.round(1))


소속사별 평균 데뷔 나이 (20명 이상):
             count  mean
Company                 
DSP             30  17.4
Pledis          29  17.8
TS              26  18.2
Fantagio        20  18.3
JYP             56  18.5
TOP Media       25  18.5
Starship        44  18.5
FNC             52  18.6
C9              21  18.7
SM              58  18.7
KQ              25  19.0
Woollim         37  19.1
Cube            37  19.1
YG              34  19.5
Jellyfish       21  20.2
RBW             21  20.5
Star Empire     21  20.5
n.CH            24  20.9


## 정리

| 목적 | 문법 |
|---|---|
| 문자열 → 날짜 (값 1개) | `datetime.strptime(s, '%d/%m/%Y')` |
| 날짜 → 문자열 | `dt.strftime('%Y-%m-%d')` |
| 문자열 → 날짜 (열 전체) | `pd.to_datetime(열, dayfirst=True, errors='coerce')` |
| 연·월·요일 추출 | `.dt.year`, `.dt.month`, `.dt.day_name()` |
| 기간 계산 | `날짜2 - 날짜1` → `.days` |
| 연도별 집계 | `df.groupby(df['날짜'].dt.year)` |

**꼭 기억할 것**
- `dd/mm/yyyy` 형식은 반드시 `dayfirst=True` (안 그러면 월/일이 뒤바뀜)
- `errors='coerce'` 를 쓰면 에러 없이 `NaT` 로 처리되므로, **변환 후 `isna().sum()` 으로 실패 건수를 꼭 확인**
- 문자열 상태로 정렬·비교하면 결과가 틀린다. 반드시 날짜형으로 변환할 것

다음: `11python_basic_streamlit.ipynb` (분석 결과를 웹 앱으로)